# 第9章: 事前学習済み言語モデル（BERT型）

本章では、BERT型の事前学習済みモデルを利用して、マスク単語の予測や文ベクトルの計算、評判分析器（ポジネガ分類器）の構築に取り組む。

## 80. トークン化

"The movie was full of incomprehensibilities."という文をトークンに分解し、トークン列を表示せよ。

In [10]:
!pip install transformers torch -q

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "The movie was full of incomprehensibilities."
tokens = tokenizer.tokenize(text)

print(tokens)

['the', 'movie', 'was', 'full', 'of', 'inc', '##omp', '##re', '##hen', '##si', '##bilities', '.']


## 81. マスクの予測

"The movie was full of [MASK]."の"[MASK]"を埋めるのに最も適切なトークンを求めよ。

In [12]:
from transformers import pipeline

# Masked Language Model
unmasker = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

text = "The movie was full of [MASK]."

results = unmasker(text)

# 最上位候補
print("Best token:", results[0]["token_str"])
print("Score:", results[0]["score"])
print("Sentence:", results[0]["sequence"])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Best token: fun
Score: 0.10711938142776489
Sentence: the movie was full of fun.


## 82. マスクのtop-k予測

"The movie was full of [MASK]."の"[MASK]"に埋めるのに適切なトークン上位10個と、その確率（尤度）を求めよ。

In [13]:
results = unmasker(text, top_k=10)

print(f"Input: {text}\n")

for rank, result in enumerate(results, start=1):
    print(f"{rank:2d}. Token      : {result['token_str']}")
    print(f"    Probability: {result['score']:.6f}")
    print(f"    Sentence   : {result['sequence']}")
    print()

Input: The movie was full of [MASK].

 1. Token      : fun
    Probability: 0.107119
    Sentence   : the movie was full of fun.

 2. Token      : surprises
    Probability: 0.066345
    Sentence   : the movie was full of surprises.

 3. Token      : drama
    Probability: 0.044684
    Sentence   : the movie was full of drama.

 4. Token      : stars
    Probability: 0.027217
    Sentence   : the movie was full of stars.

 5. Token      : laughs
    Probability: 0.025413
    Sentence   : the movie was full of laughs.

 6. Token      : action
    Probability: 0.019517
    Sentence   : the movie was full of action.

 7. Token      : excitement
    Probability: 0.019038
    Sentence   : the movie was full of excitement.

 8. Token      : people
    Probability: 0.018290
    Sentence   : the movie was full of people.

 9. Token      : tension
    Probability: 0.015031
    Sentence   : the movie was full of tension.

10. Token      : music
    Probability: 0.014646
    Sentence   : the movi

## 83. CLSトークンによる文ベクトル

以下の文の全ての組み合わせに対して、最終層の[CLS]トークンの埋め込みベクトルを用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


In [15]:
import torch
import itertools
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

cls_vectors = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    cls_vector = outputs.last_hidden_state[0, 0, :].numpy()
    cls_vectors.append(cls_vector)

for (s1, v1), (s2, v2) in itertools.combinations(
    zip(sentences, cls_vectors), 2
):
    sim = cosine_similarity([v1], [v2])[0][0]

    print(f"文1: {s1}")
    print(f"文2: {s2}")
    print(f"コサイン類似度: {sim:.4f}")
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


文1: The movie was full of fun.
文2: The movie was full of excitement.
コサイン類似度: 0.9881

文1: The movie was full of fun.
文2: The movie was full of crap.
コサイン類似度: 0.9558

文1: The movie was full of fun.
文2: The movie was full of rubbish.
コサイン類似度: 0.9475

文1: The movie was full of excitement.
文2: The movie was full of crap.
コサイン類似度: 0.9541

文1: The movie was full of excitement.
文2: The movie was full of rubbish.
コサイン類似度: 0.9487

文1: The movie was full of crap.
文2: The movie was full of rubbish.
コサイン類似度: 0.9807



## 84. 平均による文ベクトル

以下の文の全ての組み合わせに対して、最終層の埋め込みベクトルの平均を用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

In [16]:
!pip install transformers torch scikit-learn -q

import torch
import itertools
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

sentence_vectors = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    # 最終層の出力
    last_hidden_state = outputs.last_hidden_state

    # 全トークンの平均
    mean_vector = last_hidden_state.mean(dim=1).squeeze().numpy()

    sentence_vectors.append(mean_vector)

print("=== コサイン類似度 ===\n")

for (s1, v1), (s2, v2) in itertools.combinations(
    zip(sentences, sentence_vectors), 2
):
    sim = cosine_similarity([v1], [v2])[0][0]

    print(f"文1: {s1}")
    print(f"文2: {s2}")
    print(f"コサイン類似度: {sim:.4f}")
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== コサイン類似度 ===

文1: The movie was full of fun.
文2: The movie was full of excitement.
コサイン類似度: 0.9568

文1: The movie was full of fun.
文2: The movie was full of crap.
コサイン類似度: 0.8490

文1: The movie was full of fun.
文2: The movie was full of rubbish.
コサイン類似度: 0.8169

文1: The movie was full of excitement.
文2: The movie was full of crap.
コサイン類似度: 0.8352

文1: The movie was full of excitement.
文2: The movie was full of rubbish.
コサイン類似度: 0.7938

文1: The movie was full of crap.
文2: The movie was full of rubbish.
コサイン類似度: 0.9226



## 85. データセットの準備

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) から訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、さらに全てのテキストはトークン列に変換せよ。

## 86. ミニバッチの作成

85で読み込んだ訓練データの一部（例えば冒頭の4事例）に対して、パディングなどの処理を行い、トークン列の長さを揃えてミニバッチを構成せよ。

## 87. ファインチューニング

訓練セットを用い、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

## 88. 極性分析

問題87でファインチューニングされたモデルを用いて、以下の文の極性を予測せよ。

- "The movie was full of incomprehensibilities."
- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


## 89. アーキテクチャの変更

問題87とは異なるアーキテクチャ（例えば[CLS]トークンを用いるか、各トークンの最大値プーリングを用いるなど）の分類モデルを設計し、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。